<!-- paper-paired-role-banner -->
> 🟦 **실습 코드 · 왼쪽 창** — TODO를 먼저 직접 구현하세요. 막히면 오른쪽 정답의 같은 셀에서 필요한 한 줄만 확인합니다.

# 12. VAE — Auto-Encoding Variational Bayes

- **원 논문:** [Auto-Encoding Variational Bayes](https://arxiv.org/abs/1312.6114)
- **저자 / 발표:** Diederik P. Kingma and Max Welling · ICLR (2014)
- **난이도 / 예상 시간:** 중급 · 약 60분
- **선수 지식:** PyTorch 기본, 확률분포, KL divergence, 자동미분

이 노트북은 논문 결과 수치를 그대로 재현하는 대규모 benchmark가 아니라,
핵심 수식과 구조가 실제로 어떻게 동작하는지 확인하는 **하드웨어 인식 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하며, 작은 NumPy·Python 실험은
전송 비용을 피하기 위해 CPU에서 실행될 수 있습니다.

## 재현 범위와 완료 기준

**이 노트북이 재현하는 것**

2차원 혼합분포를 대상으로 재매개변수화, 닫힌형태 KL, ELBO를 직접 구현하고 작은 VAE가 입력을 복원하면서 잠재분포를 표준정규 사전에 가깝게 만드는지 확인한다.

**원 논문과 다른 것**

논문은 MNIST와 Frey Face에 확률적 인코더·디코더를 적용한다. 여기서는 다운로드 없이 512개의 2차원 점과 2차원 잠재변수로 같은 SGVB 메커니즘만 재현한다.

완료하려면 TODO를 구현하고, 검증 셀의 `assert`를 모두 통과시키고, 마지막 관찰 질문에
자신의 말로 답하세요. 수치가 논문과 다르더라도 핵심 불변식과 비교 방향이 맞으면 성공입니다.

## 논문 ↔ 노트북 지도

        절 번호와 수식 번호는 위 링크의 대표 공개본을 기준으로 합니다. 판본에 따라 페이지는 달라질 수
        있으므로 **절 제목과 수식 번호**를 함께 사용하세요.

        | 원 논문의 위치 | 이 노트북의 대응 실습 | 확인할 증거 |
        |---|---|---|
        | §2.2, Eq. (1)–(3) | `vae_objective`: reconstruction loss와 `kl_standard_normal`의 합 | ELBO의 두 항, scalar loss, finite gradient를 각각 검사 |
| §2.3, Eq. (4)–(8), Algorithm 1 | `reparameterize`와 `TinyVAE.forward`의 `z = mu + std * epsilon` | latent shape와 encoder까지 이어지는 gradient를 검사 |
| §2.4 및 §3, Eq. (9)–(10) | `TinyVAE.encode`의 `mu`, `logvar` head와 닫힌형태 KL | `mu`·`logvar` shape와 KL의 비음수성을 검사 |

        > 읽기 순서: 먼저 해당 절을 훑고 → 코드를 예측해 작성하고 → 출력이 논문의 주장과 왜
        > 연결되는지 한 문장으로 설명합니다.

## 핵심 수식과 구현 설계

$$
\log p_\theta(x)\ge
\mathbb{E}_{q_\phi(z\mid x)}[\log p_\theta(x\mid z)]
-D_{\mathrm{KL}}(q_\phi(z\mid x)\|p(z))
$$

- encoder $q_\phi$는 $\mu,\log\sigma^2$를, decoder $p_\theta$는 reconstruction을 출력합니다.
- reconstruction과 latent prior regularization의 균형이 VAE objective의 핵심입니다.
- reparameterization $z=\mu+\sigma\odot\epsilon$을 함수로 분리해 gradient 경로를 확인합니다.
- $x:[B,D_x]$에서 $\mu,\log\sigma^2,z:[B,D_z]$를 거쳐 $\hat x:[B,D_x]$로 복원합니다.

## 구현 선택 이유

- **원문에서 보존한 부분:** §2.2, Eq. (1)–(3)
- **노트북 구현:** `vae_objective`: reconstruction loss와 `kl_standard_normal`의 합
- **이렇게 구현한 이유:** 완성된 고수준 model을 한 줄로 호출하면
  논문의 핵심 shape·mask·loss·상태 변화가 숨겨집니다. 따라서 이 계산을
  이름이 드러나는 class와 함수로 나누고 중간 tensor를 assertion으로
  검사합니다.
- **실패를 잡는 증거:** ELBO의 두 항, scalar loss, finite gradient를 각각 검사를 출력 크기만 아니라
  구현 불변식으로 사용합니다.
- **축소 선택:** 논문은 MNIST와 Frey Face에 확률적 인코더·디코더를 적용한다. 여기서는 다운로드 없이 512개의 2차원 점과 2차원 잠재변수로 같은 SGVB 메커니즘만 재현한다.는 기본 실행에서 생략합니다.
  이는 Windows CPU에서도 반복 실험이 가능하게 하면서 핵심 메커니즘은
  유지하기 위한 선택입니다.

## 1. 논문의 핵심을 작은 문제로 옮기기

VAE는 관측 $x$의 로그우도 대신 다음 evidence lower bound(ELBO)를 최대화합니다.

$$\mathcal L(x)=\mathbb E_{q_\phi(z\mid x)}[\log p_\theta(x\mid z)]
-D_{KL}(q_\phi(z\mid x)\Vert p(z)).$$

이 식은 논문 §2.2 Eq. (3)에 대응합니다. §2.3–2.4의 핵심은
$z=\mu+\sigma\odot\epsilon$, $\epsilon\sim\mathcal N(0,I)$로 표본 경로를
미분 가능하게 만드는 것입니다. 원 논문의 이미지 likelihood 대신 여기서는
2차원 점의 Gaussian reconstruction error를 사용합니다.

**성공 기준:** 학습 뒤 결정론적 복원 MSE가 줄고, 전체 loss가 초기값보다 낮아야 합니다.

### 구현 단계 1 · 실행 환경·데이터 계약 (setup 예외)

- **원문 위치:** 해당 없음 — import, seed, device, 로컬 데이터 로딩만 담당하며 논문의 학습 연산을 구현하지 않습니다.
- **Tensor shape 계약:** x [B,Dₓ] → μ/logσ²/z [B,D_z] → reconstruction [B,Dₓ]
- **구현 이유:** 이후 셀의 비교가 재현 가능하도록 장치·dtype·seed와 입력 불변식을 먼저 고정합니다.
- **완료 증거:** 데이터 존재 여부, 입력 rank, 장치 선택 assertion과 환경 요약 출력이 통과해야 합니다.

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
from llm_engineering_lab.acceleration import get_accelerator

torch.manual_seed(120)
np.random.seed(120)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device
# This tiny teaching batch may be faster with AI_LAB_DEVICE=cpu;
# auto acceleration preserves the same code path for larger variants.

data_rng = torch.Generator().manual_seed(120)
centers = torch.tensor(
    [[-1.6, -1.6], [-1.6, 1.6], [1.6, -1.6], [1.6, 1.6]], dtype=torch.float32
)
component = torch.randint(0, len(centers), (512,), generator=data_rng)
points = centers[component] + 0.22 * torch.randn(512, 2, generator=data_rng)
order = torch.randperm(len(points), generator=data_rng)
points, component = points[order], component[order]
points, component = ACCELERATOR.move(points, component)
x_train, x_test = points[:384], points[384:]
y_train, y_test = component[:384], component[384:]

print(ACCELERATOR.summary())
print(f"train={tuple(x_train.shape)}, test={tuple(x_test.shape)}")
assert x_train.shape == (384, 2)
assert torch.isfinite(x_train).all()

### 구현 단계 2 · 목적함수·학습 update

- **원문 위치:** §2.3, Eq. (4)–(8), Algorithm 1 / §2.4 및 §3, Eq. (9)–(10)
- **이 셀의 책임:** `reparameterize`와 `TinyVAE.forward`의 `z = mu + std * epsilon` / `TinyVAE.encode`의 `mu`, `logvar` head와 닫힌형태 KL
- **Tensor shape 계약:** x [B,Dₓ] → μ/logσ²/z [B,D_z] → reconstruction [B,Dₓ]
- **구현 이유:** forward 결과와 논문 objective, gradient update의 책임을 분리합니다.
- **완료 증거:** latent shape와 encoder까지 이어지는 gradient를 검사 / `mu`·`logvar` shape와 KL의 비음수성을 검사. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
# TODO 1: 논문 §2.4의 z = mu + sigma * epsilon을 완성하세요.
# TODO 2: q=N(mu, diag(sigma^2)), p=N(0,I)의 샘플별 KL을 완성하세요.
def reparameterize(mu, logvar, eps=None):
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
    result = None
    assert result is not None, "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
    return result


def kl_standard_normal(mu, logvar):
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
    result = None
    assert result is not None, "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
    return result


probe_mu = torch.zeros(4, 2, requires_grad=True)
probe_logvar = torch.zeros(4, 2)
probe_z = reparameterize(probe_mu, probe_logvar, eps=torch.zeros_like(probe_mu))
assert torch.allclose(probe_z, probe_mu)
assert torch.allclose(kl_standard_normal(probe_mu, probe_logvar), torch.zeros(4))
probe_z.sum().backward()
assert probe_mu.grad is not None


def _return_shape_fixture(result):
    """바로 위 Markdown의 반환 shape 계약을 확인한다."""
    assert result is not None, "TODO 결과가 아직 비어 있습니다."
    shape = getattr(result, "shape", None)
    container = isinstance(result, (tuple, list, dict, float, int))
    assert shape is not None or container
    return result

### 구현 단계 3 · 핵심 연산·모델

- **원문 위치:** §2.3, Eq. (4)–(8), Algorithm 1 / §2.4 및 §3, Eq. (9)–(10)
- **이 셀의 책임:** `reparameterize`와 `TinyVAE.forward`의 `z = mu + std * epsilon` / `TinyVAE.encode`의 `mu`, `logvar` head와 닫힌형태 KL
- **Tensor shape 계약:** x [B,Dₓ] → μ/logσ²/z [B,D_z] → reconstruction [B,Dₓ]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** latent shape와 encoder까지 이어지는 gradient를 검사 / `mu`·`logvar` shape와 KL의 비음수성을 검사. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
# TODO: 평균/로그분산 head와 2층 decoder를 정의하세요.
class TinyVAE(nn.Module):
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""

    def __init__(self, latent_dim=2):
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        self._exercise_contract_pending = True

    def encode(self, x):
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        result = None
        assert result is not None, (
            "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
        )
        return result

    def decode(self, z):
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        result = None
        assert result is not None, (
            "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
        )
        return result

    def forward(self, x):
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        result = None
        assert result is not None, (
            "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
        )
        return result


vae = TinyVAE().to(DEVICE)
reconstruction, mu, logvar = vae(x_train[:8])
assert reconstruction.shape == (8, 2)
assert mu.shape == logvar.shape == (8, 2)


def _return_shape_fixture(result):
    """바로 위 Markdown의 반환 shape 계약을 확인한다."""
    assert result is not None, "TODO 결과가 아직 비어 있습니다."
    shape = getattr(result, "shape", None)
    container = isinstance(result, (tuple, list, dict, float, int))
    assert shape is not None or container
    return result

### 구현 단계 4 · 목적함수·학습 update

- **원문 위치:** §2.2, Eq. (1)–(3)
- **이 셀의 책임:** `vae_objective`: reconstruction loss와 `kl_standard_normal`의 합
- **Tensor shape 계약:** x [B,Dₓ] → μ/logσ²/z [B,D_z] → reconstruction [B,Dₓ]
- **구현 이유:** forward 결과와 논문 objective, gradient update의 책임을 분리합니다.
- **완료 증거:** ELBO의 두 항, scalar loss, finite gradient를 각각 검사. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
# TODO 1: reconstruction + beta * KL인 음의 ELBO를 구현하세요.
# TODO 2: Adam으로 300번의 작은 full-batch SGVB update를 수행하세요.
def vae_objective(model, batch, beta=0.1, deterministic=False):
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
    result = None
    assert result is not None, "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
    return result


optimizer = torch.optim.Adam(vae.parameters(), lr=0.01)
history = []
with torch.no_grad():
    initial_loss = float(vae_objective(vae, x_train, deterministic=True)[0])
for step in range(300):
    pass
assert len(history) == 300


def _return_shape_fixture(result):
    """바로 위 Markdown의 반환 shape 계약을 확인한다."""
    assert result is not None, "TODO 결과가 아직 비어 있습니다."
    shape = getattr(result, "shape", None)
    container = isinstance(result, (tuple, list, dict, float, int))
    assert shape is not None or container
    return result

### 구현 단계 5 · 평가·완료 증거

- **원문 위치:** §2.2, Eq. (1)–(3) / §2.4 및 §3, Eq. (9)–(10)
- **이 셀의 책임:** `vae_objective`: reconstruction loss와 `kl_standard_normal`의 합 / `TinyVAE.encode`의 `mu`, `logvar` head와 닫힌형태 KL
- **Tensor shape 계약:** x [B,Dₓ] → μ/logσ²/z [B,D_z] → reconstruction [B,Dₓ]
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** ELBO의 두 항, scalar loss, finite gradient를 각각 검사 / `mu`·`logvar` shape와 KL의 비음수성을 검사. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
# TODO: test 복원 MSE와 평균 KL을 계산하고, loss/latent/reconstruction을 그리세요.
_stage_result = None
assert _stage_result is not None, "TODO: 위 단계의 결과와 검증을 구현하세요."

## 7. 해석 TODO

1. `beta`를 0으로 만들면 reconstruction과 latent geometry가 각각 어떻게 바뀔까요?
2. `eps=0`인 평가가 무작위 표본 평가보다 안정적인 이유를 한 문장으로 쓰세요.
3. 이 실험이 논문 Eq. (8)의 미니배치 추정과 다른 점을 적으세요.

## 마무리 관찰 기록

아래 세 문장을 직접 완성하세요.

1. 이 논문의 핵심 연산은 `_____`이고, 코드에서는 `_____`에 해당한다.
2. 원 규모 실험 대신 미니 재현을 사용했기 때문에 확인할 수 없는 주장은 `_____`이다.
3. 한 가지 변수를 바꾸어 다시 실행한다면 `_____`를 바꾸고 `_____`를 측정하겠다.

**추가 실험:** seed는 유지한 채 한 변수만 바꾸고, 변경 전후의 metric을 표로 남기세요.